# 1 — Entrenar y guardar el modelo final

Entrena el modelo final desde los datos de entrada y guarda todo lo necesario para predecir en `modelo/`.

**Modelo final (dos capas):**
- **Capa 1:** regresión logística sobre los logits de tres modelos: LightGBM (features base + perfiles + 9 reglas de conocimiento),
  CatBoost (features base, `seller_id` crudo) y TabPFN v3.5 (las mismas features del LightGBM, en contexto).
- **Capa 2:** en tres zonas donde la capa 1 duda (banda, anillo, bolsillo), una fusión TabPFN combina la probabilidad de la capa 1,
  las lecturas de los LLM y las features de la fila. En la banda es TabPFN-3.5-Thinking por la API de Prior Labs (en el
  notebook 2 entra su predicción guardada); en anillo y bolsillo, TabPFN v3.5.

**Requisitos:** GPU con CUDA (TabPFN con 56.000 filas de contexto; probado en Kaggle con una T4), la variable de entorno
`TABPFN_TOKEN` y los datos en `data/` (ver README). Tiempo aproximado: 2 horas en Kaggle.

**Por qué este notebook también arma las matrices de test:** el pipeline es transductivo. Los conteos y perfiles de vendedor y
categoría se calculan sobre train + test, y TabPFN predice leyendo el train como contexto. Lo que aprende cada modelo se guarda
acá; el notebook 2 lo aplica al test.

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# Funciona corriendo desde la raíz del repositorio o desde notebooks/
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / 'src'))
CARPETA_MODELO = RAIZ / 'modelo'

from features import construir_dataset
from features_texto_grupo import features_texto_grupo
from fusiones import ZONAS, contexto_de_zona
from modelado import generar_folds, mejor_umbral
from modelo_final import (entrenar_catboost, entrenar_lightgbm, entrenar_meta, features_oof_por_fila, matrices_base,
                          matrices_perfil_conocimiento, matrices_tabpfn, oof_meta, oof_tabpfn)

CARPETA_MODELO.mkdir(exist_ok=True)

## 1. Datos y folds

CV estratificada de 5 folds (semilla 42), la misma partición para todos los modelos.

In [ ]:
X_train, X_test, target, T_train, T_test = construir_dataset()
folds = generar_folds(target)
print(f'train {X_train.shape} | test {X_test.shape} | usados en train {target.mean():.3f}')

## 2. CatBoost

Target encodings y modelos de texto (TF-IDF + regresión logística sobre título, garantía y atributos) se ajustan **dentro de cada
fold**: el OOF es honesto. El modelo final se reentrena con el 100% de train y 1.2 × el promedio de árboles de la CV.

In [ ]:
inicio = time.time()
folds_base, X_completo_base, X_test_base = matrices_base(X_train, X_test, T_train, T_test, target, folds)
oof_catboost, modelo_catboost, X_test_catboost = entrenar_catboost(folds_base, X_completo_base, X_test_base, target)
modelo_catboost.save_model(str(CARPETA_MODELO / 'catboost.cbm'))
X_test_catboost.to_pickle(CARPETA_MODELO / 'test_catboost.pkl')
print(f'CatBoost: accuracy OOF {mejor_umbral(oof_catboost, target)[1]:.5f} | {(time.time() - inicio) / 60:.0f} min')

## 3. LightGBM

Mismas matrices anidadas más perfiles de vendedor y categoría y las 9 reglas de conocimiento del mundo. Tres semillas: el OOF es
su promedio y se guarda un modelo por semilla.

In [ ]:
inicio = time.time()
folds_perfil, X_completo_perfil, X_test_perfil = matrices_perfil_conocimiento(X_train, X_test, T_train, T_test, target, folds)
oof_lightgbm, modelos_lightgbm, X_test_lightgbm = entrenar_lightgbm(folds_perfil, X_completo_perfil, X_test_perfil, target)
joblib.dump(modelos_lightgbm, CARPETA_MODELO / 'lightgbm.pkl')
X_test_lightgbm.to_pickle(CARPETA_MODELO / 'test_lightgbm.pkl')
print(f'LightGBM: accuracy OOF {mejor_umbral(oof_lightgbm, target)[1]:.5f} | {(time.time() - inicio) / 60:.0f} min')

## 4. TabPFN (capa 1)

Las features del LightGBM en formato numérico, sin los identificadores de alta cardinalidad (153 columnas). TabPFN no se entrena:
predice en contexto. Su "modelo" es el contexto (las 70.000 filas de train), que se guarda para el notebook 2.

In [ ]:
inicio = time.time()
folds_tabpfn, completo_tabpfn, indices_categoricos, nombres_columnas = matrices_tabpfn(folds_perfil, X_completo_perfil, X_test_perfil, target)
oof_tabpfn_base = oof_tabpfn(folds_tabpfn, indices_categoricos, len(target))
np.savez_compressed(CARPETA_MODELO / 'tabpfn_contexto.npz', **completo_tabpfn)
print(f'TabPFN: accuracy OOF {mejor_umbral(oof_tabpfn_base, target)[1]:.5f} | {(time.time() - inicio) / 60:.0f} min')

## 5. Meta-modelo (capa 1)

Regresión logística sobre los logits de los tres OOF. Su OOF (con los mismos folds) da el umbral del resto de las filas y es la
probabilidad de la capa 1 que reciben las fusiones.

In [ ]:
oofs = [oof_lightgbm, oof_catboost, oof_tabpfn_base]
oof_capa1, umbral_resto, accuracy_capa1 = oof_meta(oofs, target, folds)
meta = entrenar_meta(oofs, target)
joblib.dump(meta, CARPETA_MODELO / 'meta.pkl')
pesos = dict(zip(['lightgbm', 'catboost', 'tabpfn'], meta.coef_[0].round(3)))
print(f'Capa 1: accuracy OOF {accuracy_capa1:.5f} con umbral {umbral_resto:.3f} | pesos {pesos}')

## 6. Fusiones (capa 2)

Para cada zona, el contexto de TabPFN: las filas de train que leyeron los LLM, con el OOF de la capa 1, las lecturas, las 153
features calculadas sin la etiqueta de la fila (validación de su fold) y, en anillo y bolsillo, 3 features de texto por grupo.
El contexto de la banda es el mismo que recibe TabPFN-3.5-Thinking.

In [ ]:
texto_grupo_train, texto_grupo_test = features_texto_grupo(X_train, X_test, T_train, T_test)
X_train_oof = features_oof_por_fila(folds_tabpfn, len(target))
for zona in ZONAS:
    contexto = contexto_de_zona(zona, oof_capa1, X_train_oof, indices_categoricos, texto_grupo_train)
    np.savez_compressed(CARPETA_MODELO / f'fusion_{zona}.npz', X=contexto['X'], y=contexto['y'], categoricos=contexto['categoricos'])
    print(f'{zona}: contexto {contexto["X"].shape} | usados {contexto["y"].mean():.3f}')

np.save(CARPETA_MODELO / 'texto_grupo_test.npy', texto_grupo_test)
configuracion = {'umbral_resto': float(umbral_resto), 'indices_categoricos_tabpfn': indices_categoricos, 'columnas_tabpfn': nombres_columnas}
(CARPETA_MODELO / 'configuracion.json').write_text(json.dumps(configuracion, indent=1, ensure_ascii=False))
print(sorted(archivo.name for archivo in CARPETA_MODELO.iterdir()))